# Netflix Content Analysis
## A Comprehensive Data Analysis Project

**Author:** Data Analyst Portfolio Project  
**Date:** 2024  
**Dataset:** Netflix Movies and TV Shows

---

### Project Objectives:
1. Understand the distribution of content types (Movies vs TV Shows)
2. Analyze content trends over time
3. Identify top content-producing countries
4. Explore genre preferences and patterns
5. Examine rating distributions
6. Analyze content duration patterns

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

Matplotlib is building the font cache; this may take a moment.


Libraries imported successfully!


## 2. Load and Explore Dataset

In [2]:
df = pd.read_csv('../data/netflix_titles.csv')
print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

Dataset Shape: (8807, 12)

Columns: ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Title_1,NaN,"Actor_457, Actor_840, Actor_382, Actor_699, Ac...",Mexico,"August 15, 2009",1980,TV-G,173 min,"Action & Adventure, Reality TV",Description for Title_1
1,s2,TV Show,Title_2,NaN,"Actor_999, Actor_205, Actor_102, Actor_749",United States,"September 11, 2008",2003,TV-PG,7 Seasons,"Dramas, Documentaries",Description for Title_2
2,s3,TV Show,Title_3,Director_251,NaN,United States,"October 15, 2013",1989,TV-MA,7 Seasons,"Action & Adventure, Anime Series",Description for Title_3
3,s4,Movie,Title_4,Director_35,"Actor_779, Actor_118",United States,"February 13, 2016",1947,TV-14,79 min,Crime TV Shows,Description for Title_4
4,s5,Movie,Title_5,NaN,"Actor_174, Actor_265, Actor_423, Actor_506",India,"April 15, 2015",2015,TV-14,156 min,"Thrillers, Sci-Fi & Fantasy",Description for Title_5


In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Data Cleaning

In [ ]:
print("Missing Values:")
print(df.isnull().sum())
print(f"\nTotal Missing: {df.isnull().sum().sum()}")

In [ ]:
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month

df_clean = df.dropna(subset=['date_added', 'rating'])
print(f"Cleaned Dataset Shape: {df_clean.shape}")

## 4. Exploratory Data Analysis (EDA)

### 4.1 Content Type Distribution

In [ ]:
type_counts = df_clean['type'].value_counts()
print(type_counts)
print(f"\nPercentage:\n{df_clean['type'].value_counts(normalize=True) * 100}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

colors = ['#E50914', '#221f1f']
type_counts.plot(kind='bar', ax=ax1, color=colors, edgecolor='black')
ax1.set_title('Content Type Distribution', fontsize=16, fontweight='bold')
ax1.set_xlabel('Type', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.tick_params(axis='x', rotation=0)

ax2.pie(type_counts, labels=type_counts.index, autopct='%1.1f%%', 
        colors=colors, startangle=90, explode=(0.05, 0))
ax2.set_title('Content Type Proportion', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.savefig('../visualizations/content_type_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

### 4.2 Content Added Over Time

In [ ]:
yearly_content = df_clean.groupby(['year_added', 'type']).size().unstack(fill_value=0)
yearly_content.head(10)

In [ ]:
plt.figure(figsize=(14, 6))
yearly_content.plot(kind='line', marker='o', linewidth=2, markersize=6, color=['#E50914', '#221f1f'])
plt.title('Netflix Content Added Over Years', fontsize=16, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Number of Titles', fontsize=12)
plt.legend(title='Type', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../visualizations/content_over_time.png', dpi=300, bbox_inches='tight')
plt.show()

### 4.3 Top Content Producing Countries

In [ ]:
top_countries = df_clean['country'].value_counts().head(10)
print("Top 10 Content Producing Countries:")
print(top_countries)

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(x=top_countries.values, y=top_countries.index, palette='rocket')
plt.title('Top 10 Content Producing Countries', fontsize=16, fontweight='bold')
plt.xlabel('Number of Titles', fontsize=12)
plt.ylabel('Country', fontsize=12)
plt.tight_layout()
plt.savefig('../visualizations/top_countries.png', dpi=300, bbox_inches='tight')
plt.show()

### 4.4 Rating Distribution

In [ ]:
rating_counts = df_clean['rating'].value_counts()
print("Rating Distribution:")
print(rating_counts)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

rating_counts.plot(kind='bar', ax=ax1, color='#E50914', edgecolor='black')
ax1.set_title('Overall Rating Distribution', fontsize=14, fontweight='bold')
ax1.set_xlabel('Rating', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.tick_params(axis='x', rotation=45)

rating_by_type = df_clean.groupby(['type', 'rating']).size().unstack(fill_value=0)
rating_by_type.T.plot(kind='bar', ax=ax2, color=['#E50914', '#221f1f'], edgecolor='black')
ax2.set_title('Rating Distribution by Content Type', fontsize=14, fontweight='bold')
ax2.set_xlabel('Rating', fontsize=12)
ax2.set_ylabel('Count', fontsize=12)
ax2.tick_params(axis='x', rotation=45)
ax2.legend(title='Type')

plt.tight_layout()
plt.savefig('../visualizations/rating_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

### 4.5 Genre Analysis

In [ ]:
all_genres = df_clean['listed_in'].str.split(', ').explode()
top_genres = all_genres.value_counts().head(15)
print("Top 15 Genres:")
print(top_genres)

In [ ]:
plt.figure(figsize=(12, 8))
sns.barplot(y=top_genres.index, x=top_genres.values, palette='viridis')
plt.title('Top 15 Genres on Netflix', fontsize=16, fontweight='bold')
plt.xlabel('Number of Titles', fontsize=12)
plt.ylabel('Genre', fontsize=12)
plt.tight_layout()
plt.savefig('../visualizations/top_genres.png', dpi=300, bbox_inches='tight')
plt.show()

### 4.6 Release Year Analysis

In [ ]:
plt.figure(figsize=(14, 6))
df_clean[df_clean['release_year'] >= 1990]['release_year'].hist(bins=30, color='#E50914', edgecolor='black')
plt.title('Distribution of Content by Release Year (1990+)', fontsize=16, fontweight='bold')
plt.xlabel('Release Year', fontsize=12)
plt.ylabel('Number of Titles', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../visualizations/release_year_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

### 4.7 Duration Analysis

In [ ]:
movies = df_clean[df_clean['type'] == 'Movie'].copy()
movies['duration_min'] = movies['duration'].str.extract('(\d+)').astype(float)

tv_shows = df_clean[df_clean['type'] == 'TV Show'].copy()
tv_shows['seasons'] = tv_shows['duration'].str.extract('(\d+)').astype(float)

print(f"Movie Duration Stats:\n{movies['duration_min'].describe()}")
print(f"\nTV Show Seasons Stats:\n{tv_shows['seasons'].describe()}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

movies['duration_min'].hist(bins=30, ax=ax1, color='#E50914', edgecolor='black')
ax1.set_title('Movie Duration Distribution', fontsize=14, fontweight='bold')
ax1.set_xlabel('Duration (minutes)', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.axvline(movies['duration_min'].median(), color='yellow', linestyle='--', linewidth=2, label=f"Median: {movies['duration_min'].median():.0f} min")
ax1.legend()

tv_shows['seasons'].value_counts().sort_index().plot(kind='bar', ax=ax2, color='#221f1f', edgecolor='black')
ax2.set_title('TV Show Seasons Distribution', fontsize=14, fontweight='bold')
ax2.set_xlabel('Number of Seasons', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../visualizations/duration_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

### 4.8 Monthly Content Addition Pattern

In [ ]:
monthly_content = df_clean['month_added'].value_counts().sort_index()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

plt.figure(figsize=(12, 6))
plt.plot(monthly_content.index, monthly_content.values, marker='o', linewidth=2, markersize=8, color='#E50914')
plt.xticks(range(1, 13), month_names)
plt.title('Content Addition Pattern by Month', fontsize=16, fontweight='bold')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Number of Titles Added', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../visualizations/monthly_pattern.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Key Insights and Conclusions

In [ ]:
print("="*60)
print("KEY INSIGHTS FROM NETFLIX CONTENT ANALYSIS")
print("="*60)

print(f"\n1. CONTENT TYPE:")
print(f"   - Movies dominate with {(type_counts['Movie']/type_counts.sum()*100):.1f}% of content")
print(f"   - TV Shows comprise {(type_counts['TV Show']/type_counts.sum()*100):.1f}% of content")

print(f"\n2. GROWTH TREND:")
print(f"   - Peak content addition year: {yearly_content.sum(axis=1).idxmax()}")
print(f"   - Content additions show exponential growth post-2015")

print(f"\n3. GEOGRAPHIC DISTRIBUTION:")
print(f"   - Top producer: {top_countries.index[0]} ({top_countries.iloc[0]} titles)")
print(f"   - Top 3 countries produce {(top_countries.head(3).sum()/len(df_clean)*100):.1f}% of content")

print(f"\n4. CONTENT RATINGS:")
print(f"   - Most common rating: {rating_counts.index[0]} ({rating_counts.iloc[0]} titles)")
print(f"   - Mature content (TV-MA, R) represents significant portion")

print(f"\n5. GENRE PREFERENCES:")
print(f"   - Most popular genre: {top_genres.index[0]}")
print(f"   - Drama and comedy dominate content library")

print(f"\n6. DURATION PATTERNS:")
print(f"   - Average movie duration: {movies['duration_min'].mean():.0f} minutes")
print(f"   - Most TV shows have 1 season (limited series trend)")

print(f"\n7. SEASONAL PATTERNS:")
print(f"   - Peak month for additions: Month {monthly_content.idxmax()}")
print(f"   - Strategic content releases align with viewing patterns")

print("\n" + "="*60)

## 6. Business Recommendations

Based on the analysis:

1. **Content Strategy**: Continue focus on movies while expanding TV show library
2. **International Expansion**: Invest in content from emerging markets
3. **Genre Diversification**: Balance popular genres with niche content
4. **Release Timing**: Optimize content drops based on monthly patterns
5. **Duration Optimization**: Maintain 90-120 min sweet spot for movies
6. **Rating Mix**: Ensure diverse rating distribution for all demographics

## 7. Future Analysis Opportunities

- Sentiment analysis on descriptions
- Actor/Director network analysis
- Predictive modeling for content success
- Competitive analysis with other platforms
- Regional preference patterns
- Content gap identification